# Question 3 - 26. Remove Duplicates from Sorted Array

Given an integer array `nums` sorted in **non-decreasing order**, remove the duplicates **in-place** such that each unique element appears only **once**. The **relative order** of the elements should be kept the **same**. Then return *the number of unique elements in* `nums`.

Consider the number of unique elements of `nums` to be `k`, to get accepted, you need to do the following things:

- Change the array `nums` such that the first `k` elements of `nums` contain the unique elements in the order they were present in `nums` initially. The remaining elements of `nums` are not important as well as the size of `nums`.
- Return `k`.

**Custom Judge:**

    int[] nums = [...]; // Input array
    int[] expectedNums = [...]; // The expected answer with correct length

    int k = removeDuplicates(nums); // Calls your implementation

    assert k == expectedNums.length;
    for (int i = 0; i < k; i++) {
        assert nums[i] == expectedNums[i];
    }

If all assertions pass, then your solution will be accepted.

**Example 1:**

    Input: nums = [1,1,2]
    Output: 2, nums = [1,2,_]
    Explanation: Your function should return k = 2, with the first two elements of nums being 1 and 2 respectively.
    It does not matter what you leave beyond the returned k (hence they are underscores).

**Example 2:**

    Input: nums = [0,0,1,1,1,2,2,3,3,4]
    Output: 5, nums = [0,1,2,3,4,_,_,_,_,_]
    Explanation: Your function should return k = 5, with the first five elements of nums being 0, 1, 2, 3, and 4 respectively.
    It does not matter what you leave beyond the returned k (hence they are underscores).

**Constraints:**

- `1 <= nums.length <= 3 * 10^4`
- `-100 <= nums[i] <= 100`
- `nums` is sorted in **non-decreasing** order.

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Two pointers (reader / writer) on a sorted array

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (set + sort) | O(n log n) | O(n) |
| 2️⃣ Optimized (compare with last kept) | O(n) | O(1) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

You have a **sorted** list with repeats, like `[0, 0, 1, 1, 1, 2, 2, 3, 3, 4]`. You want each value to appear **just once**, in order, packed at the front: `[0, 1, 2, 3, 4, …]`. Then report how many distinct values there are (`k = 5`).

As in Question 2, the boxes are bolted down. You can't shrink the array, only rewrite what's inside it.

**The one fact that makes this easy:** the list is **sorted**, so all copies of the same number **sit right next to each other**. You never have to wonder "did I see a 2 somewhere far back?" You only ever need to compare with **the last value you kept**.

**Everyday picture:** you're reading a sorted attendance list aloud to write a clean roster. You write a name only if it's **different from the last name you wrote down**. "Alice, Alice, Bob, Bob, Bob, Carol" → you write Alice, Bob, Carol. You never need to look further back than your last entry.

### Step 0 · Clarify before coding

🗣️ *"Let me confirm: the array is sorted in non-decreasing order, so duplicates are adjacent. I modify it in place so the first k slots hold each unique value once, in the original order, and return k."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Is the input guaranteed sorted? | **Yes.** | Duplicates are neighbours, so one comparison per element is enough. |
| Must the order be preserved? | **Yes.** | Rules out tricks like swapping with the end. |
| What about after index `k`? | Ignored. | No cleanup needed. |
| Can the array be empty? | No, length ≥ 1. | The first element is always kept. |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** *"Python has a tool for removing duplicates: a **set**."* A set is a collection that automatically keeps only one copy of each value. But sets have no order, so we sort the result and write it back.

🗣️ *"The quickest correct solution is sorted(set(nums)): the set drops duplicates and sorting restores the order. Then I copy those values into the front of nums and return the count."*

**What's wrong with it?**
- **Time:** sorting costs about n × log n steps. We're re-sorting data that was **already sorted**.
- **Memory:** the set and the sorted list are extra copies of up to n values, so O(n) extra space. The problem wants in place.

🗣️ *"That's O(n log n) time and O(n) extra space. It ignores the most useful fact we were given: the input is sorted, so duplicates are already next to each other."*

In [1]:
class SolutionBrute:
    def removeDuplicates(self, nums: list[int]) -> int:
        uniq = sorted(set(nums))     # set drops repeats, sort restores order
        nums[:len(uniq)] = uniq      # write them back to the front
        return len(uniq)

### Step 2 · Optimize: the "aha" moment

> 💡 **Aha:** since the array is sorted, a value is **new** if and only if it's **different from the last value I kept**. One comparison per element is all it takes, with no set and no sorting.

This is the **same reader/writer trick as Question 2**:
- **Writer `k`**: the next free slot in the "clean" section at the front. The first element is always unique, so the clean section starts as `[nums[0]]` and `k = 1`.
- **Reader `r`**: walks from index 1 to the end.
- The last kept value is always `nums[k − 1]`.

Rule: if `nums[r] != nums[k − 1]`, it's a new value, so copy it to `nums[k]` and move `k` forward. Otherwise it's a repeat, so skip it.

🗣️ *"Because the array is sorted, duplicates are grouped together, so I only need to compare each value with the last value I kept. I keep a write pointer k, starting at 1 because the first element is always unique. I scan from index 1. Whenever nums[r] differs from nums[k − 1], the last kept value, I write it at position k and advance k. At the end, k is the number of unique values. That's one pass, O(n) time and O(1) space."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| `sorted(set(nums))` | ✅ quick first answer | Correct, but O(n log n) and uses O(n) extra memory. It ignores the sorted input. |
| Delete duplicates with `pop`/`del` | ❌ | Each delete shifts the tail, so O(n²). |
| Hash set to track "seen" values while scanning | ⚠️ | O(n) time but O(n) memory. Needed for **unsorted** input, overkill here. |
| **Compare with the last kept value (reader/writer)** | ✅ **best** | One pass, no extra memory. Uses the fact that the input is sorted. |

**Why compare with `nums[k − 1]` and not `nums[r − 1]`?** Both happen to work for this problem, because the writer never overwrites something the reader still needs. But `nums[k − 1]` says what we really mean ("the last value I **kept**"), and that's the version that generalises to Question 4.

In [2]:
class Solution:
    def removeDuplicates(self, nums: list[int]) -> int:
        k = 1                               # nums[0] is always kept
        for r in range(1, len(nums)):
            if nums[r] != nums[k - 1]:      # different from the last kept value → new
                nums[k] = nums[r]
                k += 1
        return k

### Step 3 · Toy example walkthrough (tell it like a story)

`nums = [0, 0, 1, 1, 2]`. Start with `k = 1` (the `0` at index 0 is kept).

| r | nums[r] | last kept nums[k−1] | new? | action | k | clean section nums[:k] |
|---|---|---|---|---|---|---|
| 1 | 0 | 0 | ❌ same | skip | 1 | [0] |
| 2 | 1 | 0 | ✅ | nums[1] = 1 | 2 | [0, 1] |
| 3 | 1 | 1 | ❌ same | skip | 2 | [0, 1] |
| 4 | 2 | 1 | ✅ | nums[2] = 2 | 3 | [0, 1, 2] |

Return **3**, with `nums[:3] = [0, 1, 2]` ✅

**Narrated:** "I keep the first 0. The next 0 is the same as my last kept value, so I skip it. Then a 1, which is new, so I write it in slot 1. The next 1 is a repeat, skip. Then a 2, new, written into slot 2. I've kept three values, so k is 3."

In [3]:
def run(cls, nums):
    nums = nums[:]
    k = cls().removeDuplicates(nums)
    return k, nums[:k]

cases = [
    ([1, 1, 2], (2, [1, 2])),
    ([0, 0, 1, 1, 1, 2, 2, 3, 3, 4], (5, [0, 1, 2, 3, 4])),
    ([7], (1, [7])),                    # single element
    ([-1, -1, -1], (1, [-1])),          # all the same
    ([1, 2, 3], (3, [1, 2, 3])),        # already unique
]
for nums, expected in cases:
    assert run(SolutionBrute, nums) == expected
    assert run(Solution, nums) == expected
    print(nums, "->", expected)
print("All tests passed ✅")

[1, 1, 2] -> (2, [1, 2])
[0, 0, 1, 1, 1, 2, 2, 3, 3, 4] -> (5, [0, 1, 2, 3, 4])
[7] -> (1, [7])
[-1, -1, -1] -> (1, [-1])
[1, 2, 3] -> (3, [1, 2, 3])
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| Set + sort | O(n log n) | O(n) | Copies everything and re-sorts it. |
| **Reader/writer** | **O(n)** | **O(1)** | Looks at each value once and keeps only two index numbers. |

**Why O(n) time?** One loop over the array, with one comparison and at most one copy per element. Double the input, double the work.

**Why O(1) memory?** Only `k` and `r` are stored, regardless of the input size.

**Edge cases to mention:** a single element (k = 1), all equal (k = 1), already unique (k = n, nothing moves).

**Likely follow-up:** *"What if each value may appear at most **twice**?"* → That's Question 4. Change one character: compare with `nums[k − 2]` instead of `nums[k − 1]`. In general, "at most `t` copies" means comparing with `nums[k − t]`.

---

#### 🗣️ Full interview script (about 60 seconds)

*"The array is sorted, and I need each value to appear once at the front, in order, returning how many unique values there are.*

*The quick solution is sorted(set(nums)) written back into the array. That's O(n log n) time and O(n) space, and it ignores the fact that the input is already sorted.*

*Because it's sorted, duplicates are adjacent, so a value is new exactly when it differs from the last value I kept. I use a write pointer k starting at 1, since the first element is always unique. I scan from index 1; whenever nums[r] differs from nums[k − 1], I copy it to nums[k] and increment k. At the end, k is the answer.*

*One pass, O(n) time, and O(1) extra space."*